# DeFFSCUTE - Deductive, Few-Shot Survey Classification Using Text Embeddings

**Halvor Tyseng, Jonas Timmann Mjaaland, Markus Fleten Kreutzer and Tor Ole B. Odden**
Center for Computing in Science Education and Center for Interdisciplinary Education, University of Oslo, Norway.

**Disclaimer:** Usage of the method and code below will require a citation of this comptuational essay. Citation details pertaining to this research are found in the CITATION.cff file.

MIT License

Copyright (c) 2025 halvorty

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

## Introduction 

This notebook present the framework proposed in the article and *Classifying open-ended survey responses with text embeddings*, that laverages properties of embedding in order to preform analysis of open-ended survey response, at scale. The framework has five main steps, and this notebook can be used to replicate the results found in the article. The embedding models, dataset and centroids can be swapped to explore.
In addition how sensitive the results are to diffent centroids selection, is estimated in the samplign section

### Table of contents

[Imports](#imports) 

**Framework steps**
1. [Data Preperation](#1-data-preperation)
2. [Embedding](#2-embedding)
3. [Category Mapping](#3-category-mapping)
4. [Classification](#4-classification)
5. [Evaluation](#5-evaluation)


[Sampling](#sampling)



## Imports 
First we import the python packeges needed.
We use the ```sentence_transformer``` library to fetch embedding models.

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score, cohen_kappa_score, matthews_corrcoef

## 1. Data preperation

First we load the dataset with text we want classified. In this case it's students responses to a physics question. This dataset is already labled by human qualitative researchers, and the coloumn "code" will serve as a benchmark for the proposed framework. The centroids dataset is similar to the sources, but it only consist a of handfull texts, that are meant to represent the categories in the embedding space.

In [2]:
# Load the data
df_sources = pd.read_excel('data/sources_amended.xlsx')
df_centroids = pd.read_excel('data/centroids.xlsx')

The sources dataset has been **selectively** coded by the qualitative researches, meaning some responses are labeled as other ("O"). To measure performance of our framework in an **exhaustive** coding task, it is here possible to exclude the other category.

In [3]:
# Remove other (noise) labeled responses - for exhaustive coding
# Set exclude_other to False if you want to include the 'Other' responses
exclude_other = True

if exclude_other:
    df_sources = df_sources[df_sources["code"] != "O"]
    df_centroids = df_centroids[df_centroids["code"] != "O"]

In [4]:
df_sources.head()

,Response,code
0,Imprecise/faulty measuring tools,L
1,air fluctuations,L
2,Slight differences in experimental setup: tabl...,L
3,The material of ramp and ball and the coeffici...,L
4,Materials issues (ball; ramps are identical),L


In [5]:
df_centroids.head()

,text,code
0,Randomness of Brownian motion,P
1,Nature of Brownian motion results in a natural...,P
2,Spins should be random,P
3,Quantum effects in electrons,P
4,The diffraction of the electron wave function ...,P


## 2. Embedding

To vectorize the text we use open source embedding models, made available on Hugging Face. 
The Sentence Transformer library makes it easy to download a text embedding model, or use a locally stored one.

Some models benefit from a prompt describing the task, or a task specification that decides a LoRA adapter to be used. The model name, what prompt or task to use, and other specificities for each model can be found in the **model card** at Hugging Face. For example the model card for intfloat/multilingual-e5-large-instruct is found at [https://huggingface.co/intfloat/multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct). We recommend using the model as intended by the provider.

In [6]:
# Choose a model from Hugging Face
model_name = 'intfloat/multilingual-e5-large-instruct'

# Set the prompt for the model, it differs based on the model used
prompt = "Instruct: Retrieve semantically similar text \nQuery: "

# Load the model with the SentenceTransformer class
model = SentenceTransformer(model_name, 
                            local_files_only=False,
                            ) 

In [ ]:
# We then use the model loaded to vectorize the text in the dataframes
# The embeddings are stored in a new column called 'embeddings'
embeddings_sources = model.encode(df_sources["Response"].values.tolist(), 
                                  prompt=prompt, 
                                  normalize_embeddings=True)
df_sources["embeddings"] = embeddings_sources.tolist()

# We also vectorize the text to be used as centroids for the categories
# Which is loaded in the df_centroids dataframe
embeddings_centroids = model.encode(df_centroids["text"].values.tolist(), 
                                    prompt=prompt, 
                                    normalize_embeddings=True)
df_centroids["embeddings"] = embeddings_centroids.tolist()

## 3. Category mapping

In this step we use the selected texts, found in the centroids dataframe, to create one centroid-vector for each category. We do this by averaging over the embeddings for all centroid texts related to a category.

In [8]:
def create_centroids(df: pd.DataFrame,
                     code_col: str = "code",
                     emb_col: str = "embeddings",) -> tuple[np.ndarray, list]:
    """
    Compute one centroid per code by averaging row embeddings.

    Returns
    -------
    centroids : (C, D) array
        L2-normalized centroids stacked in the same order as `labels`.
    labels : list
        Sorted unique codes corresponding to rows in `centroids`.
    """
    if code_col not in df or emb_col not in df:
        raise KeyError(f"DataFrame must contain '{code_col}' and '{emb_col}' columns.")

    # Ensure deterministic order
    labels = sorted(df[code_col].unique().tolist())

    centroids: list[np.ndarray] = []
    for code in labels:
        # Stack all embeddings for this code
        embs = df.loc[df[code_col] == code, emb_col]
        # Support either arrays in .values or in Python lists
        stacked = np.vstack(embs)
        centroids.append(stacked.mean(axis=0))

    centroids_arr = np.asarray(centroids)

    return centroids_arr, labels

# Build centroids
centroids, labels = create_centroids(df_centroids, code_col="code", emb_col="embeddings")

## 4. Classification

To classify a given response from the dataset we want to analyze - ```df_sources``` in our case, we measure the similarity between the embedding of the response and the centroids created in step 3. We then assign the label of that centroid who has the highest similarity. 

In [9]:
def predict_codes(embeddings: np.ndarray,
                  centroids: np.ndarray,
                  labels: list) -> list:
    """
    Assign each embedding the label of its most similar centroid (cosine).

    Parameters
    ----------
    embeddings : (N, D) array
    centroids  : (C, D) array (assumed L2-normalized row-wise)
    labels     : list of length C mapping centroid index -> code label
    """

    # Normalize so cosine similarity is dot product
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
    centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

    S = embeddings @ centroids.T  # (N, C) cosine similarities

    closest_arg = np.argmax(S, axis=1)
    return [labels[i] for i in closest_arg]

# Predict codes for new embeddings (shape: N x D)
pred_codes = predict_codes(embeddings_sources, centroids, labels)
pred_codes[:10]  # Show first 10 predictions

['L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L']

## 5. Evaluation

We evaluate the predicted labels against labels coded by the qualitative researchers.

In [10]:
def compute_metrics(y_true: np.ndarray | list, y_pred: np.ndarray | list) -> dict[str, float]:
    """
    Compute agreement/quality metrics for categorical predictions.
    """
    return {
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted")),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }

# Evaluate against ground truth in df_sources
metrics = compute_metrics(df_sources["code"].values, pred_codes)
print(f"Kappa: {metrics['kappa']:.4f}, F1: {metrics['f1_weighted']:.4f}, MCC: {metrics['mcc']:.4f}")

Kappa: 0.8279, F1: 0.9648, MCC: 0.8289


## Sampling

One way to check stability for the centroids is to sample a subset of the centroids and evaluate how much the coding consistency metrics (Kappa, F1, MCC) change. 

In [11]:
def sample_df_centroid(df: pd.DataFrame, 
                 sampling_per: float = 0.75, 
                 code_col: str = "code") -> pd.DataFrame:
    """
    Sample a subset of codes based on the specified sampling percentage.
    """
    code_counts = df[code_col].value_counts().to_dict()
    sampled_df = pd.DataFrame(columns=df.columns)
    
    for code, count in code_counts.items():
        n_samples = int(count * sampling_per)
        sampled_df = pd.concat([sampled_df, df[df[code_col] == code].sample(n=n_samples)])
    return sampled_df.reset_index(drop=True)

In [12]:
n_runs = 100

df_results = pd.DataFrame(columns=["kappa", "f1_weighted", "mcc"])

for i in range(n_runs):
    sampled_df = sample_df_centroid(df_centroids, sampling_per=0.75, code_col="code")
    # Build centroids for the sampled data
    centroids_sampled, labels_sampled = create_centroids(sampled_df,
                                                        code_col="code",
                                                        emb_col="embeddings")
    
    # Predict codes for the original embeddings using the sampled centroids
    pred_codes_sampled = predict_codes(embeddings_sources, centroids_sampled, labels_sampled)
    df_results.loc[i] = compute_metrics(df_sources["code"].values, pred_codes_sampled)

# Calculate the mean and standard deviation of the metrics
mean_metrics = df_results.mean()
std_metrics = df_results.std()

print(f"Mean Kappa: {mean_metrics['kappa']:.4f} ± {std_metrics['kappa']:.4f}")
print(f"Mean F1: {mean_metrics['f1_weighted']:.4f} ± {std_metrics['f1_weighted']:.4f}")
print(f"Mean MCC: {mean_metrics['mcc']:.4f} ± {std_metrics['mcc']:.4f}")


Mean Kappa: 0.7952 ± 0.0274
Mean F1: 0.9580 ± 0.0058
Mean MCC: 0.7973 ± 0.0259
